In [1]:
import numpy as np
from gensim.models import Word2Vec
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.model_selection import train_test_split

In [2]:
with open("../data/my_sweet_orange_tree.txt", "r", encoding="utf-8") as file:
    raw_text = file.read()

In [3]:
# tokenization

In [4]:
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/markjake/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/markjake/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/markjake/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [5]:
sentences = sent_tokenize(raw_text.lower())
tokenized_sentences = [word_tokenize(sentence) for sentence in sentences]

In [6]:
print(f"sentences size: {len(sentences)}")
print(f"token size: {len(tokenized_sentences)}")

sentences size: 4716
token size: 4716


In [7]:
embedding_model = Word2Vec(sentences=tokenized_sentences, vector_size=100, window=5, min_count=1, sg=1, workers=4, epochs=10)

In [8]:
len(embedding_model.wv.index_to_key)

3971

In [9]:
# def create_sequence(sentences):
#     pairs = []

#     for sentence in sentences:
#         for i in range(1, len(sentence)):
#             context = sentence[:i]
#             target= sentence[i]

#             pairs.append((context, target))

#     return pairs

In [10]:

WINDOW_SIZE = 3
EMBED_SIZE = embedding_model.wv.vector_size

def create_sequence(sentences, window_size=WINDOW_SIZE):
    pairs = []

    for sentence in sentences:
        for i in range(1, len(sentence)):
            context = sentence[max(0, i - window_size):i]
            target = sentence[i]
            pairs.append((context, target))

    return pairs


In [11]:
# def context_vector(context, embedding_model):
#     vectors = [
#         embedding_model.wv[word]
#         for word in context
#         if word in embedding_model.wv
#     ]

#     if len(vectors) == 0:
#         return np.zeros(100)
    
#     return np.mean(vectors, axis=0)

In [12]:
WINDOW_SIZE = 3
EMBED_SIZE = embedding_model.wv.vector_size

def context_to_vector(context, embedding_model):
    context = context[-WINDOW_SIZE:]

    vectors = [
        embedding_model.wv[word]
        if word in embedding_model.wv
        else np.zeros(EMBED_SIZE)
        for word in context
    ]

    # Pad missing positions on the left
    while len(vectors) < WINDOW_SIZE:
        vectors.insert(0, np.zeros(EMBED_SIZE))

    return np.concatenate(vectors)

In [13]:
pairs = create_sequence(tokenized_sentences)

In [14]:
word_to_idx = {
    word: idx
    for idx, word in enumerate(embedding_model.wv.index_to_key)
}

In [15]:
idx_to_word = {
    idx: word
    for word, idx in word_to_idx.items()
}

In [16]:
X = np.array([
    context_to_vector(context, embedding_model)
    for context, target in pairs
])
y = np.array([
    target
    for _, target in pairs
])

y = np.array([
    word_to_idx[word]
    for word in y
])

In [17]:
X = X.astype(np.float32)

In [18]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)

In [19]:
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

In [20]:
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"y_val shape: {y_val.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (41262, 300)
y_train shape: (41262,)
X_val shape: (5158, 300)
y_val shape: (5158,)
X_test shape: (5158, 300)
y_test shape: (5158,)


In [21]:
def onehot(Y):
    return np.eye(len(word_to_idx))[Y]

In [22]:
# relu
def relu(Z):
    return np.maximum(0, Z)
# softmax
def softmax(Z):
    exp_z = np.exp(Z - np.max(Z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

In [23]:
def initialize_parameters(
    input_size, hiddenLayer1_size, hiddenLayer2_size, output_size
):
    np.random.seed(42)

    parameters = {
        "w1": (np.random.randn(input_size, hiddenLayer1_size) * np.sqrt(2 / input_size)).astype(np.float32),
        "b1": np.zeros((1, hiddenLayer1_size)).astype(np.float32),
        "w2": (np.random.randn(hiddenLayer1_size, hiddenLayer2_size) * np.sqrt(2 / hiddenLayer1_size)).astype(np.float32),
        "b2": np.zeros((1, hiddenLayer2_size)).astype(np.float32),
        "w3": (np.random.randn(hiddenLayer2_size, output_size) * np.sqrt(2 / hiddenLayer2_size)).astype(np.float32),
        "b3": np.zeros((1, output_size)).astype(np.float32),
    }

    return parameters

In [24]:
def forward_propagation(X, parameters):
    w1, b1, w2, b2, w3, b3 = (
        parameters["w1"],
        parameters["b1"],
        parameters["w2"],
        parameters["b2"],
        parameters["w3"],
        parameters["b3"],
    )

    # first layer
    Z1 = np.dot(X, w1) + b1
    A1 = relu(Z1)

    # second layer
    Z2 = np.dot(A1, w2) + b2
    A2 = relu(Z2)

    # output layer
    Z3 = np.dot(A2, w3) + b3
    A3 = softmax(Z3)

    cache = {
        "Z1": Z1,
        "A1": A1,
        "Z2": Z2,
        "A2": A2,
        "Z3": Z3,
        "A3": A3,
    }

    return A3, cache


In [25]:
def compute_cost(Y, A3):
    m = Y.shape[0]

    correct_prob = np.clip(A3[np.arange(m), Y], 1e-12, 1.0)

    L = -1 / m * np.sum(np.log(correct_prob))

    return L

In [26]:
def backpropagation(w1, b1, w2, b2, w3, b3, cache, X, Y, alpha):
    m = Y.shape[0]
    Z1, A1, Z2, A2, Z3, A3 = cache["Z1"], cache["A1"], cache["Z2"], cache["A2"], cache["Z3"], cache["A3"]

    # output layer
    dZ3 = A3 - onehot(Y)
    dW3 = np.dot(A2.T, dZ3) / m
    db3 = np.sum(dZ3, axis=0, keepdims=True) / m

    # second layer
    dA2 = np.dot(dZ3, w3.T)
    dZ2 = dA2 * (Z2 > 0) # relu'
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m
    dW2 = np.dot(A1.T, dZ2) / m

    # first layer
    dA1 = np.dot(dZ2, w2.T)
    dZ1 = dA1 * (Z1 > 0)
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m
    dW1 = np.dot(X.T, dZ1) / m

    return dW1, db1, dW2, db2, dW3, db3

In [27]:
def backpropagation(w1, b1, w2, b2, w3, b3, cache, X, Y):
    m = Y.shape[0]

    Z1, A1 = cache["Z1"], cache["A1"]
    Z2, A2 = cache["Z2"], cache["A2"]
    A3 = cache["A3"]

    # Output layer: softmax + cross-entropy
    dZ3 = A3 - onehot(Y)
    # dZ3 = A3.copy()
    # dZ3[np.arange(len(Y)), Y] -= 1
    dW3 = A2.T @ dZ3 / m
    db3 = np.sum(dZ3, axis=0, keepdims=True) / m

    # Hidden layer 2: ReLU
    dA2 = dZ3 @ w3.T
    dZ2 = dA2 * (Z2 > 0)
    dW2 = A1.T @ dZ2 / m
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m

    # Hidden layer 1: ReLU
    dA1 = dZ2 @ w2.T
    dZ1 = dA1 * (Z1 > 0)
    dW1 = X.T @ dZ1 / m
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m

    return dW1, db1, dW2, db2, dW3, db3

In [28]:
# input_size = embedding_model.wv.vector_size * 3
# hidden1_size = 128
# hidden2_size = 64
# output_size = len(embedding_model.wv.index_to_key)

# params = initialize_parameters(input_size, hidden1_size, hidden2_size, output_size)

In [29]:
# normal gradient
# alpha = 0.05
# costs = []
# for i in range(500):
#     A3, cache = forward_propagation(X, params)
#     cost = compute_cost(y, A3)
#     costs.append(cost)
#     dw1, db1, dw2, db2, dw3, db3 = backpropagation(params["w1"], params["b1"], params["w2"], params["b2"], params["w3"], params["b3"], cache, X, y, alpha)

#     # update params
#     params["w1"] -= alpha * dw1
#     params["b1"] -= alpha * db1
    
#     params["w2"] -= alpha * dw2
#     params["b2"] -= alpha * db2
    
#     params["w3"] -= alpha * dw3
#     params["b3"] -= alpha * db3

#     if i % 50 == 0:
#         print(f"epoch {i}, lost: {cost}")

In [30]:
# # gradient descent with velocity (momentum)
# # testing

# alpha = 0.05
# beta = 0.9

# v = {
#     "dw1": np.zeros_like(params["w1"]),
#     "db1": np.zeros_like(params["b1"]),
#     "dw2": np.zeros_like(params["w2"]),
#     "db2": np.zeros_like(params["b2"]),
#     "dw3": np.zeros_like(params["w3"]),
#     "db3": np.zeros_like(params["b3"]),
# }


# costs = []
# for i in range(500):
#     A3, cache = forward_propagation(X, params)
#     cost = compute_cost(y, A3)
#     costs.append(cost)
#     dw1, db1, dw2, db2, dw3, db3 = backpropagation(params["w1"], params["b1"], params["w2"], params["b2"], params["w3"], params["b3"], cache, X, y, alpha)


#     v["dw1"] = beta * v["dw1"] + (1 - beta) * dw1
#     v["db1"] = beta * v["db1"] + (1 - beta) * db1
#     v["dw2"] = beta * v["dw2"] + (1 - beta) * dw2
#     v["db2"] = beta * v["db2"] + (1 - beta) * db2
#     v["dw3"] = beta * v["dw3"] + (1 - beta) * dw3
#     v["db3"] = beta * v["db3"] + (1 - beta) * db3
#     # update params using velocity
#     params["w1"] -= alpha * v["dw1"]
#     params["b1"] -= alpha * v["db1"]
    
#     params["w2"] -= alpha * v["dw2"]
#     params["b2"] -= alpha * v["db2"]
    
#     params["w3"] -= alpha * v["dw3"]
#     params["b3"] -= alpha * v["db3"]

#     if i % 50 == 0:
#         print(f"epoch {i}, lost: {cost}")

In [31]:
# my potato machine can't handle the training

# import numpy as np

# # Start with fresh parameters
# params = initialize_parameters(
#     X_train.shape[1], 128, 128, len(word_to_idx)
# )
# print("X:", X_train.shape)
# print("w1:", params["w1"].shape)
# print("b1:", params["b1"].shape)
# print("w2:", params["w2"].shape)
# print("b2:", params["b2"].shape)
# print("w3:", params["w3"].shape)
# print("b3:", params["b3"].shape)


# # Adam settings
# learning_rate = 0.001
# beta1 = 0.9
# beta2 = 0.999
# epsilon = 1e-8
# weight_decay = 0.01

# keys = ["w1", "b1", "w2", "b2", "w3", "b3"]

# m = {k: np.zeros_like(params[k]) for k in keys}
# v = {k: np.zeros_like(params[k]) for k in keys}

# costs = []
# epochs = 1000

# for epoch in range(1, epochs + 1):
#     A3, cache = forward_propagation(X_train, params)
#     cost = compute_cost(y_train, A3)
#     costs.append(cost)

#     grads = backpropagation(
#         params["w1"], params["b1"],
#         params["w2"], params["b2"],
#         params["w3"], params["b3"],
#         cache, X_train, y_train
#     )

#     grads = dict(zip(keys, grads))

#     for k in keys:
#         m[k] = beta1 * m[k] + (1 - beta1) * grads[k]
#         v[k] = beta2 * v[k] + (1 - beta2) * grads[k]**2

#         m_hat = m[k] / (1 - beta1**epoch)
#         v_hat = v[k] / (1 - beta2**epoch)

#         # adaptive learning rate
#         # params[k] -= learning_rate * m_hat / (
#         #     np.sqrt(v_hat) + epsilon
#         # )
#         # apply weight decay only on weights matrices
#         if k in ["w1", "w2", "w3"]:
#             params[k] *= (1 - learning_rate * weight_decay)
            
#         params[k] -= learning_rate * m_hat / (
#             np.sqrt(v_hat) + epsilon
#         )
        

#     if epoch == 1 or epoch % 100 == 0:
#         print(f"Epoch {epoch}: loss = {cost:.4f}")

# # # Evaluate training accuracy
# # A3, _ = forward_propagation(X, params)
# # predictions = np.argmax(A3, axis=1)

# # print("Training accuracy:", np.mean(predictions == y))

# # unique, counts = np.unique(predictions, return_counts=True)
# # top = np.argsort(counts)[::-1][:10]

# # print("\nMost predicted words:")
# # for i in top:
# #     idx = unique[i]
# #     print(f"{idx_to_word[idx]}: {counts[i]} ({counts[i]/len(y):.1%})")

In [32]:
params = initialize_parameters(
    X_train.shape[1], 128, 128, len(word_to_idx)
)

print("X:", X_train.shape)
print("w1:", params["w1"].shape)
print("b1:", params["b1"].shape)
print("w2:", params["w2"].shape)
print("b2:", params["b2"].shape)
print("w3:", params["w3"].shape)
print("b3:", params["b3"].shape)

learning_rate = 0.001
beta1 = 0.9
beta2 = 0.999
epsilon = 1e-8
weight_decay = 0.001

keys = ["w1", "b1", "w2", "b2", "w3", "b3"]

m_adam = {k: np.zeros_like(params[k]) for k in keys}
v_adam = {k: np.zeros_like(params[k]) for k in keys}

epochs = 30
batch_size = 128
step = 0

for epoch in range(1, epochs + 1):
    # shuffle training examples each epoch
    indices = np.random.permutation(len(X_train))

    epoch_loss = 0.0

    for start in range(0, len(X_train), batch_size):
        batch_idx = indices[start:start + batch_size]

        X_batch = X_train[batch_idx]
        y_batch = y_train[batch_idx]

        # Forward propagation
        A3, cache = forward_propagation(X_batch, params)

        batch_loss = compute_cost(y_batch, A3)
        epoch_loss += batch_loss * len(batch_idx)

        # Backpropagation
        grads = backpropagation(
            params["w1"], params["b1"],
            params["w2"], params["b2"],
            params["w3"], params["b3"],
            cache, X_batch, y_batch
        )

        grads = dict(zip(keys, grads))
        step += 1

        # Adam update
        for k in keys:
            g = grads[k]

            m_adam[k] = beta1 * m_adam[k] + (1 - beta1) * g
            v_adam[k] = beta2 * v_adam[k] + (1 - beta2) * (g ** 2)

            m_hat = m_adam[k] / (1 - beta1 ** step)
            v_hat = v_adam[k] / (1 - beta2 ** step)

            # Decay weights, not biases
            if k in ["w1", "w2", "w3"]:
                params[k] *= (1 - learning_rate * weight_decay)

            params[k] -= learning_rate * m_hat / (
                np.sqrt(v_hat) + epsilon
            )

    avg_loss = epoch_loss / len(X_train)

    print(f"Epoch {epoch}: loss = {avg_loss:.4f}")

X: (41262, 300)
w1: (300, 128)
b1: (1, 128)
w2: (128, 128)
b2: (1, 128)
w3: (128, 3971)
b3: (1, 3971)
Epoch 1: loss = 4.5060
Epoch 2: loss = 3.7852
Epoch 3: loss = 3.5590
Epoch 4: loss = 3.4117
Epoch 5: loss = 3.3006
Epoch 6: loss = 3.2022
Epoch 7: loss = 3.1112
Epoch 8: loss = 3.0263
Epoch 9: loss = 2.9489
Epoch 10: loss = 2.8740
Epoch 11: loss = 2.8044
Epoch 12: loss = 2.7441
Epoch 13: loss = 2.6876
Epoch 14: loss = 2.6374
Epoch 15: loss = 2.5896
Epoch 16: loss = 2.5479
Epoch 17: loss = 2.5119
Epoch 18: loss = 2.4731
Epoch 19: loss = 2.4393
Epoch 20: loss = 2.4084
Epoch 21: loss = 2.3800
Epoch 22: loss = 2.3534
Epoch 23: loss = 2.3283
Epoch 24: loss = 2.3034
Epoch 25: loss = 2.2801
Epoch 26: loss = 2.2574
Epoch 27: loss = 2.2356
Epoch 28: loss = 2.2174
Epoch 29: loss = 2.1985
Epoch 30: loss = 2.1800


In [34]:
print("Output shape:", A3_train.shape)
print("Training labels shape:", y_train.shape)
print("Validation labels shape:", y_val.shape)

print("Training label range:",
      np.min(y_train), np.max(y_train))

print("Validation label range:",
      np.min(y_val), np.max(y_val))

print("Output min:", np.min(A3_train))
print("Output max:", np.max(A3_train))
print("Mean output row sum:", np.mean(np.sum(A3_train, axis=1)))

NameError: name 'A3_train' is not defined

In [58]:
A3_val, _ = forward_propagation(X_val, params)
val_pred = np.argmax(A3_val, axis=1)

print("Validation accuracy:", np.mean(val_pred == y_val))
print("Validation loss:", compute_cost(y_val, A3_val))
print()

A3_train, _ = forward_propagation(X_train, params)
train_pred = np.argmax(A3_train, axis=1)

print("train accuracy:", np.mean(train_pred == y_train))
print("train loss:", compute_cost(y_train, A3_train))
print()

A3_test, _ = forward_propagation(X_test, params)
test_pred = np.argmax(A3_test, axis=1)

print("test accuracy:", np.mean(test_pred == y_test))
print("test loss:", compute_cost(y_test, A3_test))

Validation accuracy: 0.22741372625048467
Validation loss: 6.6495113

train accuracy: 0.3745334690514275
train loss: 2.5488014

test accuracy: 0.22566886390073673
test loss: 6.6646442


In [ ]:
def predict_next_word(context):
    context = context.lower().split()

    if not context:
        raise ValueError("Provide at least one context word.")

    # Check that all context words exist in the vocabulary
    unknown = [word for word in context if word not in word_to_idx]
    if unknown:
        raise ValueError(f"Unknown vocabulary words: {unknown}")

    x = context_to_vector(context, embedding_model).reshape(1, -1)
    probabilities, _ = forward_propagation(x, params)

    predicted_idx = int(np.argmax(probabilities[0]))
    return idx_to_word[predicted_idx]


# print(predict_next_word("sometimes"))
# print(predict_next_word("he was"))
# print(predict_next_word("it was a"))

In [ ]:
def generate_text(start_text, num_words=50):
    words = start_text.lower().split()

    for _ in range(num_words):
        context = words[-3:]
        next_word = predict_next_word(" ".join(context))
        words.append(next_word)

    return " ".join(words)

print(generate_text("he", 50))

In [ ]:
embedding_model.wv["baby"]

In [26]:
def show_top_predictions(context, top_n=5):
    words = context.lower().split()

    unknown = [w for w in words if w not in word_to_idx]
    if unknown:
        print("Unknown words:", unknown)
        return

    x = context_to_vector(words, embedding_model).reshape(1, -1)
    probabilities, _ = forward_propagation(x, params)

    top_indices = np.argsort(probabilities[0])[::-1][:top_n]

    print(f"\nContext: {context}")
    for idx in top_indices:
        print(
            f"{idx_to_word[idx]:<12} "
            f"{probabilities[0][idx]:.4f}"
        )

show_top_predictions("the")
show_top_predictions("he was")
show_top_predictions("it was a")
show_top_predictions("once upon a time")


Context: the
desultory    0.0991
mere         0.0966
course       0.0885
sensation    0.0800
strouds      0.0778

Context: he was
the          0.0980
,            0.0538
i            0.0533
it           0.0529
--           0.0402

Context: it was a
vocation     0.2379
villa        0.2170
sketch       0.1779
failure      0.0764
note         0.0320

Context: once upon a time
under        0.2616
dubarry_     0.2470
being        0.0682
oak          0.0465
have         0.0452
